In [5]:
# Closed systems testing is done to check if model generates biomass or energy when there 
# is no external supply of metabolites
import cobra

In [13]:
model = cobra.io.read_sbml_model("Helicobacter_pylori.xml")

'' is not a valid SBML 'SId'.


In [15]:
# normal FBA before closing the system
solution = model.optimize()

print("Status:", solution.status)
print("Objective value:", solution.objective_value)

Status: optimal
Objective value: 20.23558895922464


In [17]:
# closing all exchange reactions
closed_model = model.copy()

for rxn in closed_model.reactions:
    if rxn.id.startswith("EX_"):
        rxn.lower_bound = 0
        rxn.upper_bound = 0

print("Exchange reactions closed.")

Exchange reactions closed.


In [21]:
closed_solution = closed_model.optimize()

print("Closed-system status:", closed_solution.status)
print("Closed-system objective:", closed_solution.objective_value)

Closed-system status: optimal
Closed-system objective: 0.0


In [25]:
# energy generation check
for rxn in model.reactions:
    if "atp" in rxn.id.lower() or "atp" in rxn.name.lower():
        print(rxn.id, ":", rxn.name, ":", rxn.reaction)

ADK7 : adenylate kinase (dATP) : amp[c] + datp[c] <=> adp[c] + dadp[c]
ATPS4 : ATP synthase (four protons for one ATP) : adp[c] + 4.0 h[e] + pi[c] --> atp[c] + h2o[c] + 3.0 h[c]
Cut1 : Copper export via ATPase : atp[c] + cu2[c] + h2o[c] --> adp[c] + cu2[e] + h[c] + pi[c]
DGK1 : Deoxyguanylate Kinase (dGMP:ATP) : atp[c] + dgmp[c] <=> adp[c] + dgdp[c]
DM_atp_c_ : Demand for ATP, Cytosolic : atp[c] + h2o[c] --> adp[c] + h[c] + pi[c]
GK1 : Guanylate Kinase (GMP:ATP) : atp[c] + gmp[c] <=> adp[c] + gdp[c]
GK2 : guanylate kinase (GMPdATP) : datp[c] + gmp[c] <=> dadp[c] + gdp[c]
HEX1 : Hexokinase (D-Glucose:ATP) : atp[c] + glc_D[c] --> adp[c] + g6p[c] + h[c]
HEXb : ATPD-glucose 6-phosphotransferase : atp[c] + glc_bD[c] --> adp[c] + g6p_B[c] + h[c]
HMPK1 : hydroxymethylpyrimidine kinase (ATP) : 4ahmmp[c] + atp[c] --> 4ampm[c] + adp[c] + h[c]
NDPK1 : Nucleoside-Diphosphate Kinase (ATP:GDP) : atp[c] + gdp[c] <=> adp[c] + gtp[c]
NDPK2 : Nucleoside-Diphosphate Kinase (ATP:UDP) : atp[c] + udp[c] <=>

In [27]:
egc_model = model.copy()

# Close all exchange reactions
for rxn in egc_model.reactions:
    if rxn.id.startswith("EX_"):
        rxn.lower_bound = 0
        rxn.upper_bound = 0

# Set ATP demand as the objective
egc_model.objective = "DM_atp_c_"

# Maximize ATP production
egc_solution = egc_model.optimize()

print("Status:", egc_solution.status)
print("ATP energy-generating flux:", egc_solution.objective_value)

Status: optimal
ATP energy-generating flux: 0.0


In [29]:
print("Current objective:")
print(model.objective)

print("\nObjective direction:")
print(model.objective.direction)

solution = model.optimize()

print("\nFBA status:", solution.status)
print("Objective value:", solution.objective_value)

Current objective:
Maximize
1.0*biomass525 - 1.0*biomass525_reverse_5c178

Objective direction:
max

FBA status: optimal
Objective value: 20.23558895922475
